In [ ]:
# 添加目录到系统路径方便导入模块，该项目的根目录为".../machine-learning-toy-code"
import sys
from pathlib import Path
curr_path = str(Path().absolute())       # 获取当前工作目录
parent_path = str(Path().absolute().parent)  # 获取上级目录
p_parent_path = str(Path().absolute().parent.parent)  # 获取上上级目录（项目根目录）
sys.path.append(p_parent_path)  # 将项目根目录加入系统路径，方便后续导入自定义模块
print(f"主目录为：{p_parent_path}")

In [2]:
# 下面为sci-kit版本

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml

# 加载 MNIST 手写数字数据集（70000张28x28灰度图像，共784个特征）
# MNIST 是机器学习领域的经典基准数据集，常用于分类算法的评估
mnist = fetch_openml('mnist_784')
X, y = mnist['data'], mnist['target']  # X 为像素特征矩阵，y 为数字标签(0-9)
# 按照经典划分：前60000张作为训练集，后10000张作为测试集
X_train = np.array(X[:60000], dtype=float)
y_train = np.array(y[:60000], dtype=float)
X_test = np.array(X[60000:], dtype=float)
y_test = np.array(y[60000:], dtype=float)

print(X_train.shape)  # (60000, 784)：60000个样本，每个784维
print(y_train.shape)
print(X_test.shape)   # (10000, 784)：10000个测试样本
print(y_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic 回归模型：使用 Sigmoid 函数将线性组合映射到 (0,1) 区间
# 数学形式：P(y=1|x) = sigmoid(w^T * x + b) = 1 / (1 + exp(-(w^T * x + b)))
# penalty="l1"：使用 L1 正则化，可产生稀疏权重向量（部分权重为0），实现特征选择
# solver="saga"：使用 SAGA 优化算法，适合大规模数据集和 L1 正则化
# tol=0.1：收敛容差，控制迭代停止条件
clf = LogisticRegression(penalty="l1", solver="saga", tol=0.1)
clf.fit(X_train, y_train)  # 在训练集上拟合模型
score = clf.score(X_test, y_test)  # 在测试集上计算准确率
print("Test score with L1 penalty: %.4f" % score)

In [ ]:
# 下面为pytorch版本

In [ ]:
# 导入 PyTorch 及相关库，用于加载 MNIST 数据集和可视化
from torch.utils.data import DataLoader  # 数据加载器，支持批量加载和打乱
from torchvision import datasets          # 包含常用视觉数据集（如 MNIST、CIFAR-10）
import torchvision
import torchvision.transforms as transforms  # 数据预处理和增强工具
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report  # 分类评估报告（精确率、召回率、F1值）
import numpy as np

In [ ]:
# 通过 torchvision 加载 MNIST 数据集（与上面 fetch_openml 等价，但来自 PyTorch 生态）
# transforms.ToTensor() 将 PIL 图像转换为 [0,1] 范围的 Tensor
train_dataset = datasets.MNIST(root = p_parent_path+'/datasets/', train = True,transform = transforms.ToTensor(), download = False)
test_dataset = datasets.MNIST(root = p_parent_path+'/datasets/', train = False, 
                               transform = transforms.ToTensor(), download = False)

# 设置 batch_size 为整个数据集大小，一次性加载全部数据
batch_size = len(train_dataset)
# shuffle=True 表示每个 epoch 打乱数据顺序，有助于模型训练的随机性
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)
# 使用迭代器获取一个批次的数据（此处即为全部数据）
X_train,y_train = next(iter(train_loader))
X_test,y_test = next(iter(test_loader))
# 打印前100张图片进行可视化，直观查看数据集内容
images, labels= X_train[:100], y_train[:100] 
# 使用 images 生成宽度为10张图的网格，方便展示多张图片
img = torchvision.utils.make_grid(images, nrow=10)
# cv2.imshow() 的格式是 (size1,size1,channels)，而 img 的格式是 (channels,size1,size1)
# 所以需要使用 .transpose() 转换，将颜色通道数放至第三维
img = img.numpy().transpose(1,2,0)
print(images.shape)   # (100, 1, 28, 28)：100张单通道28x28图像
print(labels.reshape(10,10))  # 按10x10网格排列显示标签
print(img.shape)
plt.imshow(img)
plt.show()

In [ ]:
# 将 PyTorch Tensor 转换为 NumPy 数组，以便使用 scikit-learn 的模型
# .cpu() 确保数据在 CPU 上（如果之前在 GPU 上需要先转移）
X_train,y_train = X_train.cpu().numpy(),y_train.cpu().numpy()
X_test,y_test = X_test.cpu().numpy(),y_test.cpu().numpy()

In [ ]:
# 将图像数据从 (N, 1, 28, 28) 的4维张量展平为 (N, 784) 的2维矩阵
# Logistic 回归需要一维特征向量作为输入，784 = 28 * 28（图像像素总数）
X_train = X_train.reshape(X_train.shape[0],784)
X_test = X_test.reshape(X_test.shape[0],784)

In [ ]:
# 多分类 Logistic 回归（默认使用 OvR 策略：One-vs-Rest，一对多）
# solver="lbfgs"：拟牛顿法（L-BFGS），一种近似二阶优化方法，收敛速度快
# max_iter=400：最大迭代次数，保证模型有足够迭代次数收敛
model = LogisticRegression(solver='lbfgs', max_iter=400)
model.fit(X_train, y_train)  # 在训练集上训练模型
y_pred = model.predict(X_test)  # 在测试集上进行预测
# classification_report 输出每个类别的精确率(precision)、召回率(recall)和 F1 分数
# 精确率 = TP/(TP+FP)，召回率 = TP/(TP+FN)，F1 = 2*P*R/(P+R)
print(classification_report(y_test, y_pred)) # 打印报告

In [ ]:
# 为数据添加偏置列（全1列），等价于模型中的截距项 b
# 添加后特征变为 [x1, x2, ..., x784, 1]，使 w^T * x 可以直接包含偏置
ones_col=[[1] for i in range(len(X_train))] # 生成全为1的二维嵌套列表，即[[1],[1],...,[1]]
X_train = np.append(X_train,ones_col,axis=1)  # axis=1 表示按列拼接
x_train = np.mat(X_train)  # 转换为 NumPy 矩阵，便于后续矩阵运算
X_test = np.append(X_test,ones_col,axis=1)
x_test = np.mat(X_test)
# 将多分类问题转化为二分类问题：判断是否为数字"1"
# 标签为1的样本标记为正类(1)，其余标记为负类(0)
# 这是 Logistic 回归最基本的二分类应用场景
y_train=np.array([1 if y_train[i]==1 else 0 for i in range(len(y_train))])
y_test=np.array([1 if y_test[i]==1 else 0 for i in range(len(y_test))])

In [ ]:
# 二分类 Logistic 回归：判断图片是否为数字"1"
# solver="lbfgs"：拟牛顿法优化器
# 与上面的多分类相比，二分类任务更简单，准确率通常更高
model = LogisticRegression(solver='lbfgs', max_iter=100)
model.fit(X_train, y_train)  # 训练模型
y_pred = model.predict(X_test)  # 预测
print(classification_report(y_test, y_pred)) # 打印报告